# Exercise 09

## Changes to the Evacuation Model

To enable exercises about stochasticity, the evacuation model has been modified:

* Isolated stochastic processes
  * *rng_propagate* for information propagation
  * *rng_placement* for placing agents in the room
  * *rng_orientation* for the agents' initial and random (in case crowd prediction does not change orientation) orientation
* Abandon `set()`-implementation in `move_toward_target()` to ensure deterministic randomness
* Correct `batchrun_model()` to apply rng for batch scenarios.


## Evaluation Code


In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import itertools
import time
from datetime import timedelta
from matplotlib.colors import LinearSegmentedColormap

import sys
sys.path.insert(0,'../../abmodel')

from fire_evacuation.model import FireEvacuation, FireEvacuationScenario
from fire_evacuation.agent import Human

unikcolors = [np.array((80,149,200))/255, np.array((74,172,150))/255,
                                                  np.array((234,195,114))/255, np.array((199,16,92))/255]
uniks = LinearSegmentedColormap.from_list( 'unik', unikcolors)

%matplotlib inline

In [2]:
def batchrun_model(
        scenario,
        experiments,
        replications=20,
        aggregate_output_per_run=True,
    ):
    """
    Instantiates Scenarios for parameter/replication variations, runs the model
    for every Scenario and concats results in a pd.DataFrame.

    Parameters
    ----------
    experiments: pd.DataFrame or dict
        data frame of parameter combinations with parameter names as column names
    replications: int
        number of runs with different RNG
    max_step: int
        the number of steps the model is run in case it does not terminate earlier
    aggregate_output_per_run: bool
        calculate mean values (except for Step)
    """
    # unfortunately, scenario.from_dataframe() is a class method and does not consider values in scenario,
    # so we need to add them to experiments manually:
    experiments = {key: [value] for key, value in scenario.__dict__.items()} | experiments
    
    # create combinations of given parameter values
    if type(experiments) is pd.DataFrame:
        experiments = pd.DataFrame(itertools.product(*[experiments[col] for col in experiments.columns]),
                               columns=experiments.columns)
    elif type(experiments) is dict:
        experiments = pd.DataFrame(itertools.product(*experiments.values()),
                               columns=experiments.keys())
    
    scenarios = scenario.from_dataframe(
            experiments=experiments,
            replications=replications,
            rng=scenario.rng,
    )
    results = pd.DataFrame()
    
    print(f"Perform {len(scenarios)} model runs...")
    start = time.time()
    for scenario in scenarios:
        model =  FireEvacuation(scenario=scenario)
        model.run_model()
        d = model.datacollector.get_model_vars_dataframe()
        d.insert(0, "sid", scenario.scenario_id)
        d.insert(0, "rid", scenario.replication_id)
        d.index.name="Step"
        d = d.reset_index()
        # calc means
        if aggregate_output_per_run:
            d = d.groupby(["sid", "rid"]).agg(
                {column:"mean" if column != "Step" else "max" for column in d.columns if column not in ["sid", "rid"]}
            )
        d = pd.concat([pd.DataFrame(scenario.__dict__, index=d.index), d], axis=1)
        results = pd.concat([results, d])
    print(f"Done! Took {str(timedelta(seconds=time.time() - start))} h")
    return results

# Task 2 (Stochasticity in the evacuation model)

## Subtask 2 (Isolation of random processes)

There are many random processes involved in the evacuation model. To get an idea about their particular impact on the model, underlying random number streams can be isolated, i.e. for each process a separate random number stream can be used.

Boxplots are a suitable type of figure to show the variation in results. Complete the function `plot_boxplots()` and visualise the data from simulations with varying random seed for different random processes as boxplot showing **mean** and outliers.

Execute the following code and discuss the results. Choose suitable values for `ylim` (consider plotting twice with and without outliers). Formulate statements about the impact of each of the random processes (<200 words)! Also consider outliers!

In [ ]:
# Complete the following method with the creation of boxplots:
def plot_boxplot(data=None, title="", **kwargs):
    """
    Parameters
        ----------
        data : dataFrame
            with entries "Placement", "Orientation", "Propagation" in column 'RNG' for one value of interact_moore
        title : str
    """
    
    # Implement boxplot here

    plt.title(title)

In [ ]:
results_placement = batchrun_model(
    scenario = FireEvacuationScenario(
        floor_size=14,
        human_count=100,
        nervousness_mean = 0.5,
        alarm_believers_prop = 0.7,
        facilitators_percentage= 5,
        seed_orientation = 0,
        seed_propagate = 0,
        rng=7,
    ),
    experiments = {
        "interact_moore": {0.05, 0.1, 0.5},
        "seed_placement": range(0,50),
    },
    replications = 1,
    aggregate_output_per_run = True,
)

In [ ]:
results_orientation = batchrun_model(
    scenario = FireEvacuationScenario(
        floor_size=14,
        human_count=100,
        nervousness_mean = 0.5,
        alarm_believers_prop = 0.7,
        facilitators_percentage= 20,
        seed = 0,
        seed_placement = 0,
        seed_propagate = 0,
        rng=7,
    ),
    experiments = {
        "interact_moore": {0.05, 0.1, 0.5},
        "seed_orientation": range(0,50),
    },
    replications = 1,
    aggregate_output_per_run = True,
)

In [ ]:
results_propagate = batchrun_model(
    scenario = FireEvacuationScenario(
        floor_size=14,
        human_count=100,
        nervousness_mean = 0.5,
        facilitators_percentage= 20,
        seed = 0,
        seed_placement = 0,
        seed_orientation = 0,
        rng=7,
    ),
    experiments = {
        "interact_moore": {0.05, 0.1, 0.5},
        "seed_propagate": range(0,50),
    },
    replications = 1,
    aggregate_output_per_run = True,
)

In [ ]:
data_placement = pd.DataFrame(results_placement)[['interact_moore', 'Step']].round(decimals=2)
data_orientation = pd.DataFrame(results_orientation)[['interact_moore','Step']].round(decimals=2)
data_propagate = pd.DataFrame(results_propagate)[['interact_moore','Step']].round(decimals=2)

# depending on the type of figure you're going to generate, the data may need to be organised differently!
data_placement['RNG']='Placement'
data_orientation['RNG']='Orientation'
data_propagate['RNG']='Propagation'
datas = pd.concat([data_placement, data_orientation, data_propagate], axis=0)

plot_boxplot(datas[datas['interact_moore'] == 0.05].drop(['interact_moore'], axis=1),
            title = "Variation in Steps for Random Processes for an Interaction Prob. of 0.05", showfliers=False)
plot_boxplot(datas[datas['interact_moore'] == 0.1].drop(['interact_moore'], axis=1),
            title = "Variation in Steps for Random Processes for an Interaction Prob. of 0.1", showfliers=False)
plot_boxplot(datas[datas['interact_moore'] == 0.5].drop(['interact_moore'], axis=1),
            title = "Variation in Steps for Random Processes for an Interaction Prob. of 0.5", showfliers=False)

**Describe your findings here (max. 200 words)**

## Subtask 3 (Contingency)

Contingency means the dependence of random processes on the context. Find two stochastic processes in the evacuation model that can be modelled with contingency. Provide pseudo code for each of the processes that introduces contingency. Argue for your approach!


| Property | Probability |
|----------|-------------|
| adult    | high        |
| child    | low         |


**Write your answer here (200 words max. and pseudo code)!**

## Subtask 4 (Random to deterministic)

If one wants to avoid random numbers in the evacuation model, how could the propagation of information (alarm situation) be modelled in more detail without random processes? Provide pseudo code! What are the advantages and disadvantages of a deterministic approach in general and of your approach in particular?

Are there other processes that could be modelled deterministic and detailed instead of randomly? List at least two and state the purpose(s) (simplification, uncertainty, extrapolation) for random processes for the selected ones!

**Write your anwser here (300 words max. and pseudo code)**